# Reto 1 · Perfiles ciudadanos de participación

Notebook único con todo lo resuelto hasta ahora para la convocatoria «Hackea la participación» (INE-DECEyEC), eje 1. Pregunta guía: *¿qué tipos de ciudadanía existen según su comportamiento de participación?*

**Cómo leerlo.** Cada sección explica qué hace, por qué se hace así y qué decisión de la spec (`specs/001-perfiles-ciudadanos/`) cubre. La lógica no está aquí: vive en los módulos de `src/` (con pruebas en `src/tests/`) y el notebook solo los llama y muestra el resultado.

| Sección | Qué responde | Requisitos |
|---|---|---|
| 1. Configuración | ¿Qué elecciones, rutas y umbrales se usan? | RNF-03 |
| 2. Ingesta | ¿Cómo pasan los 192 CSV a un formato rápido y compacto? | RF-01, RF-03 |
| 3. Calidad | ¿Son confiables los datos? | RF-02, RF-15 |
| 4. Indicadores | ¿Cómo se mide la participación y qué es una «celda»? | RF-04 a RF-06 |
| 5. Segmentación | ¿Cuántos perfiles hay y qué tan estables son? | RF-07 a RF-09 |
| 6. Perfiles | ¿Quiénes son y dónde viven? | RF-10 |
| 7. Sensibilidad (D16) | ¿Los perfiles son personas o son estados? | solidez |

**Limitaciones declaradas.** La fuente no trae escolaridad; trae `SEXO`, no género; las secciones cambian de límites entre elecciones; el CCPC excluye casillas especiales y residentes en el extranjero.

## 1. Configuración

Fija las constantes del análisis y las carpetas. Los CSV originales solo se leen, nunca se modifican (RNF-03). Todo lo que se publica sale de las tablas que se guardan en `entrega/tablas/` (RNF-06).

- **Elecciones:** 2009 a 2024, seis en total. Presidenciales: 2012, 2018 y 2024. Intermedias: 2009, 2015 y 2021. Se separan porque la participación es mucho mayor en las presidenciales y eso no debe confundirse con tendencia (RF-06).
- **Umbral de lista nominal:** 100 personas por celda y elección (D7).
- **Semilla:** fija, para que las mismas entradas den siempre los mismos perfiles (RF-08).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # el notebook se ejecuta desde src/

import numpy as np
import polars as pl

import analisis, calidad, indicadores, ingesta, perfiles, segmentacion
from configuracion import *

print("Elecciones:", ELECCIONES, "| semilla:", SEMILLA, "| umbral LN:", UMBRAL_LN)

Elecciones: [2009, 2012, 2015, 2018, 2021, 2024] | semilla: 2026 | umbral LN: 100


## 2. Ingesta

**Qué hace.** Localiza los 192 CSV (6 elecciones × 32 entidades), valida que todos tengan el mismo esquema de 16 columnas y escribe **un Parquet por elección** en `datos_procesados/`.

**Por qué.** Los CSV pesan unos 3.5 GB; el Parquet con tipos compactos es mucho más rápido de leer y cabe en memoria (RNF-04). Si un archivo trae otro esquema, el proceso falla con un error explícito en lugar de mezclar datos. La sección sola no es única en el país, por eso se conservan `EDOCVE` (entidad) y `SECCION` juntas (RF-03).

Si los Parquet ya existen se omiten; para regenerarlos, `REGENERAR = True`.

In [2]:
REGENERAR = False

por_anio = ingesta.listar_por_anio(BRUTOS)
assert sorted(por_anio) == ELECCIONES, "faltan elecciones"
assert all(len(r) == 32 for r in por_anio.values()), "cada elección debe traer 32 entidades"
PROCESADOS.mkdir(exist_ok=True)
ingesta.escribir_parquet_por_anio(por_anio, PROCESADOS, REGENERAR)
TABLAS.mkdir(parents=True, exist_ok=True)
catalogo = ingesta.catalogo_entidades(por_anio[ELECCIONES[-1]])   # nombres de entidad para los textos
assert catalogo.height == 32
catalogo.write_csv(TABLAS / "entidades_catalogo.csv")

pl.scan_parquet(PROCESADOS / "ccpc_*.parquet").group_by("AELEC").agg(
    pl.len().alias("filas"), pl.col("EDOCVE").n_unique().alias("entidades"), pl.col("LN").sum().alias("LN")
).sort("AELEC").collect()

2009 ya existe, se omite
2012 ya existe, se omite
2015 ya existe, se omite
2018 ya existe, se omite
2021 ya existe, se omite
2024 ya existe, se omite


AELEC,filas,entidades,LN
i16,u32,u32,i32
2009,8706843,32,77481831
2012,8604456,32,76490962
2015,9106059,32,83563462
2018,9251122,32,89123997
2021,9430502,32,93532133
2024,9773730,32,98330348


## 3. Calidad de los datos

**Qué hace.** Comprueba tres cosas y arma la tabla de calidad por elección y entidad:

1. **Identidad `LN = SV + NV + NS`** (lista nominal = votaron + no votaron + sin dato). Las filas que no cumplen se reportan, no se descartan en silencio.
2. **Duplicados** por (elección, entidad, sección, sexo, edad).
3. **Tabla de calidad:** filas, totales, porcentaje de `NS` y edades atípicas.

**Por qué importa.** `NS` (sin dato de voto) varía mucho entre elecciones; si no se controla, distorsiona las comparaciones. Por eso la participación se define sin `NS` (sección 4).

In [3]:
archivos = sorted(PROCESADOS.glob("ccpc_*.parquet"))
datos = pl.scan_parquet(archivos)

desbalance = calidad.desbalance(datos)
repetidas = calidad.duplicadas_por_archivo(archivos)
print("Filas con desbalance LN != SV+NV+NS:", int(desbalance["filas_con_desbalance"].sum()) if desbalance.height else 0)
print("Claves duplicadas:", repetidas.height)

tabla_calidad = calidad.tabla(datos)
TABLAS.mkdir(parents=True, exist_ok=True)
tabla_calidad.write_csv(TABLAS / "tabla_calidad.csv")
nacional_calidad = calidad.nacional(tabla_calidad)
nacional_calidad.write_csv(TABLAS / "calidad_nacional.csv")
nacional_calidad

Filas con desbalance LN != SV+NV+NS: 0
Claves duplicadas: 0


AELEC,LN,SV,NV,NS,part_SV_sobre_LN,part_SV_sobre_SV_NV,pct_NS
i16,i64,i64,i64,i64,f64,f64,f64
2009,77481831,33316974,42298963,1865894,43.0,44.06,2.41
2012,76490962,47383154,28939159,168649,61.95,62.08,0.22
2015,83563462,36926612,41517874,5118976,44.19,47.07,6.13
2018,89123997,51784399,31249856,6089742,58.1,62.37,6.83
2021,93532133,46389189,43099624,4043320,49.6,51.84,4.32
2024,98330348,54899228,37105466,6325654,55.83,59.67,6.43


La tabla muestra la participación nacional con dos definiciones (D1): `SV/LN` y `SV/(SV+NV)`. La segunda es la principal. **Pendiente (RF-15):** contrastarla con las cifras de la plataforma CCPC; una diferencia frente a los cómputos distritales es esperada porque el CCPC excluye casillas especiales y residentes en el extranjero.

## 4. Indicadores y celdas demográficas

**Qué hace.** Convierte las filas (sección × sexo × edad) en la unidad de análisis: la **celda demográfica** = grupo de edad × sexo × tipo de sección × entidad (D2b). Esto da perfiles de *personas* («jóvenes de zona urbana…»), que es lo que pide la convocatoria, en vez de perfiles de secciones.

Pasos y decisiones:
- **Edades:** se agrupan en tramos de 18-19, 20-24, … y todo lo de 90 o más va a «90+» (D6, RF-04). Se reporta cuántas personas afecta.
- **Participación** = `SV / (SV + NV)` como cociente de **sumas**, nunca promedio de porcentajes (D1, RF-05).
- **Celdas pequeñas:** se excluyen las que tengan menos de 100 personas en alguna elección o no existan en las seis (D7) y se reportan aparte.
- **Participación relativa** = tasa de la celda menos la tasa **de su propia entidad** en esa elección (D16b, RF-06). Así las elecciones presidenciales no se confunden con «más participación» de un grupo, y tampoco el efecto de cada estado (por ejemplo, elecciones locales concurrentes): quedan solo las diferencias entre grupos demográficos dentro de una misma entidad.

In [4]:
ind = analisis.paso_indicadores()
print("Celdas (celda × elección):", ind["celdas"].height)
print("Celdas excluidas por tamaño:", ind["excluidas"].height)
print(f"Lista nominal excluida: {ind['pct_ln_excluida']:.3f} %")
ind["atipicas"]

Celdas (celda × elección): 17046
Celdas excluidas por tamaño: 89
Lista nominal excluida: 0.003 %


AELEC,LN_90_o_mas
i16,i64
2009,384719
2012,266093
2015,355126
2018,406339
2021,460871
2024,478283


In [5]:
ind["nacional"]

AELEC,tasa_nacional
i16,f64
2009,0.440612
2012,0.620833
2015,0.47074
2018,0.623657
2021,0.518387
2024,0.596706


La tabla anterior es la participación nacional **después** de filtrar celdas pequeñas (referencia de apoyo, sección 7). El código comprueba que la suma ponderada de las diferencias sea 0 en cada elección y entidad (RF-06); si no, el paso falla.

## 5. Segmentación

**Qué hace.** Agrupa las celdas por **comportamiento**, no por demografía (la demografía sirve después para describir). Cada celda se resume en tres rasgos que separan el *nivel* de la *forma* en el tiempo (RF-07):

1. **nivel:** participación relativa media en las seis elecciones.
2. **brecha:** media en presidenciales menos media en intermedias (qué tanto «sube» un grupo cuando la elección es presidencial).
3. **tendencia:** pendiente de la participación relativa a lo largo del tiempo.

Se usa **k-means ponderado por lista nominal** con semilla fija (RF-08), probando 4, 5 y 6 perfiles. Para elegir k (RF-09): *silhouette* (separación; no admite pesos, se calcula sin ellos) y **estabilidad** (ARI entre corridas con otras semillas y con submuestras del 80 %). Se elige el k de mayor *silhouette* entre los estables (ARI medio de 0.80 o más); la interpretabilidad la confirma el equipo.

In [6]:
seg = analisis.paso_segmentacion()
print("Correlaciones entre rasgos:\n", seg["correlaciones"].round(2))
print("k elegido:", seg["k"])
seg["seleccion"]

Correlaciones entre rasgos:
 [[ 1.   -0.15 -0.  ]
 [-0.15  1.    0.24]
 [-0.    0.24  1.  ]]
k elegido: 5


k,inercia,silhouette_sin_pesos,ARI_semillas_media,ARI_semillas_min,ARI_submuestras_media,ARI_submuestras_min
i64,f64,f64,f64,f64,f64,f64
4,7.9546e7,0.196354,0.974833,0.961301,0.889518,0.834578
5,6.6692e7,0.236538,0.982913,0.970006,0.939734,0.893912
6,5.8205e7,0.225376,0.994544,0.981965,0.91835,0.72923


## 6. Caracterización de los perfiles

**Qué hace.** Describe cada perfil con cifras agregadas (sin datos personales, RF-13): peso (% de la lista nominal 2024), rasgos de comportamiento, participación por elección, edad, sexo, tipo de sección, entidades donde más pesa y la **composición de cada sección** (qué proporción de su lista pertenece a cada perfil; no una etiqueta única, RF-10).

**Cómo leer las tablas.**
- *Peso y rasgos:* nivel negativo = participa menos que el promedio nacional; brecha positiva = sube más en elecciones presidenciales.
- *Edad:* se da la **mediana** de edad y la distribución completa.
- *Entidades:* índice > 1 significa que el perfil pesa más en esa entidad que en el país.

In [7]:
per = analisis.paso_perfiles()
per["peso"]

perfil,celdas,LN_2024,nivel,brecha,tendencia,volatilidad,pct_lista_nominal_2024
i8,u32,i64,f64,f64,f64,f64,f64
1,473,33099553,-0.08894,0.02314,-0.002889,0.020245,33.66
2,441,9262754,-0.084241,-0.043647,-0.014408,0.035923,9.42
3,606,29527931,0.055247,0.001918,-0.003349,0.014354,30.03
4,592,12311345,0.079651,-0.012094,0.013307,0.033401,12.52
5,729,14124129,0.113884,-0.044052,-0.004936,0.028943,14.36


In [8]:
per["participacion"]

perfil,2009,2012,2015,2018,2021,2024
i8,f64,f64,f64,f64,f64,f64
1,0.3542,0.5506,0.3593,0.5581,0.41,0.5081
2,0.3976,0.5396,0.422,0.5214,0.436,0.4496
3,0.4995,0.6806,0.5272,0.6679,0.5621,0.6443
4,0.4863,0.6665,0.5722,0.6859,0.6377,0.7102
5,0.5657,0.7247,0.6096,0.7209,0.6515,0.703


In [9]:
print(per["sexo"]); print(per["tipo"]); per["mediana"]

shape: (5, 3)
┌────────┬─────────────┬─────────────┐
│ perfil ┆ pct_hombres ┆ pct_mujeres │
│ ---    ┆ ---         ┆ ---         │
│ i8     ┆ f64         ┆ f64         │
╞════════╪═════════════╪═════════════╡
│ 1      ┆ 55.48       ┆ 44.52       │
│ 2      ┆ 77.03       ┆ 22.97       │
│ 3      ┆ 38.11       ┆ 61.89       │
│ 4      ┆ 32.15       ┆ 67.85       │
│ 5      ┆ 46.13       ┆ 53.87       │
└────────┴─────────────┴─────────────┘
shape: (5, 4)
┌────────┬───────┬───────┬───────┐
│ perfil ┆ M     ┆ R     ┆ U     │
│ ---    ┆ ---   ┆ ---   ┆ ---   │
│ i8     ┆ f64   ┆ f64   ┆ f64   │
╞════════╪═══════╪═══════╪═══════╡
│ 1      ┆ 4.51  ┆ 0.16  ┆ 95.33 │
│ 2      ┆ 38.45 ┆ 59.37 ┆ 2.18  │
│ 3      ┆ 12.53 ┆ 4.09  ┆ 83.37 │
│ 4      ┆ 14.86 ┆ 23.08 ┆ 62.06 │
│ 5      ┆ 15.83 ┆ 53.88 ┆ 30.29 │
└────────┴───────┴───────┴───────┘


perfil,grupo_edad_mediano
i8,str
1,"""25-29"""
2,"""25-29"""
3,"""45-49"""
4,"""65-69"""
5,"""55-59"""


In [10]:
per["edad"]

perfil,18-19,20-24,25-29,30-34,35-39,40-44,45-49,50-54,55-59,60-64,65-69,70-74,75-79,80-89,90+
i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,3.73,24.2,24.13,22.66,16.01,6.34,1.15,0.37,0.31,0.14,0.0,0.07,0.05,0.51,0.35
2,6.33,26.53,22.58,14.53,10.55,7.8,4.18,1.52,0.96,0.33,0.0,0.0,0.04,1.92,2.72
3,4.53,0.51,1.63,2.33,8.27,17.64,22.18,20.3,14.32,5.3,1.24,0.65,0.55,0.53,0.0
4,0.99,1.83,2.66,6.87,4.22,2.02,1.58,1.23,1.68,14.63,21.48,15.37,11.57,12.97,0.89
5,2.0,3.33,1.77,2.68,4.01,6.63,8.75,12.46,15.99,16.58,11.32,8.45,4.81,1.2,0.0


In [11]:
per["top"]

perfil,EDOCVE,LN_2024,pct,pct_lista_nominal_2024,indice
i8,i8,i64,f64,f64,f64
1,9,3193363,40.34,33.66,1.2
1,25,892240,37.8,33.66,1.12
1,5,908078,37.67,33.66,1.12
1,3,230463,37.59,33.66,1.12
1,28,1053619,36.98,33.66,1.1
…,…,…,…,…,…
5,9,2471194,31.22,14.36,2.17
5,2,748642,23.42,14.36,1.63
5,24,461229,21.1,14.36,1.47


In [12]:
print(f"Lista nominal 2024 sin perfil (celdas excluidas): {per['ln_sin_perfil']:,} de {per['ln_total']:,}")
print("Secciones con composición calculada:", per["comp"].height)
print("Entidades con más de 80 % de su lista en un solo perfil:", per["entidades_dominantes"], "de 32")

Lista nominal 2024 sin perfil (celdas excluidas): 4,636 de 98,330,348
Secciones con composición calculada: 70750
Entidades con más de 80 % de su lista en un solo perfil: 0 de 32


## 7. Sensibilidad (D16): referencia nacional frente a referencia por entidad

**El problema que motivó D16.** Con la referencia nacional (la D4a original) los perfiles se parecen a etiquetas de estado: algunos cubren casi toda la lista de una entidad. Si todas las celdas de una entidad caen en el mismo perfil, la demografía no decide la asignación. La medida es η² (qué fracción de la varianza de un rasgo explica la entidad).

**La decisión.** Se adoptó la referencia por entidad como corrida principal (D16b). Esta sección repite el análisis con la referencia nacional como análisis de apoyo y compara ambas; sus tablas están en `entrega/tablas/sensibilidad_nacional/`.

**Hipótesis no verificada:** la `brecha` podría reflejar elecciones locales concurrentes (distintas en cada entidad). Comprobarlo exige el calendario local, que no está en los datos.

In [13]:
analisis.paso_indicadores("nacional")
seg_n = analisis.paso_segmentacion("nacional")
per_n = analisis.paso_perfiles("nacional")

def varianza_explicada_por_entidad(referencia):
    sufijo = rutas(referencia)["sufijo"]
    a = pl.read_parquet(PROCESADOS / f"celdas_perfil{sufijo}.parquet")
    w, ent, salida = a["LN_medio"].to_numpy(), a["EDOCVE"].to_numpy(), {}
    for rasgo in ["nivel", "brecha", "tendencia"]:
        x = a[rasgo].to_numpy()
        total = np.sum(w * (x - np.average(x, weights=w)) ** 2)
        dentro = sum(np.sum(w[ent == e] * (x[ent == e] - np.average(x[ent == e], weights=w[ent == e])) ** 2) for e in np.unique(ent))
        salida[rasgo] = round(1 - dentro / total, 3)
    return salida

eta_e, eta_n = varianza_explicada_por_entidad("entidad"), varianza_explicada_por_entidad("nacional")
comparacion = pl.DataFrame({
    "referencia": ["entidad (principal, D16b)", "nacional (apoyo, D4a)"],
    "k": [seg["k"], seg_n["k"]],
    "entidades_con_80pct_en_un_perfil": [per["entidades_dominantes"], per_n["entidades_dominantes"]],
    **{f"eta2_{r}": [eta_e[r], eta_n[r]] for r in ["nivel", "brecha", "tendencia"]},
})
comparacion.write_csv(TABLAS / "sensibilidad_comparacion.csv")
comparacion

referencia,k,entidades_con_80pct_en_un_perfil,eta2_nivel,eta2_brecha,eta2_tendencia
str,i64,i64,f64,f64,f64
"""entidad (principal, D16b)""",5,0,0.0,0.001,0.004
"""nacional (apoyo, D4a)""",4,10,0.166,0.744,0.759


In [14]:
per_n["peso"]

perfil,celdas,LN_2024,nivel,brecha,tendencia,volatilidad,pct_lista_nominal_2024
i8,u32,i64,f64,f64,f64,f64,f64
1,777,29871215,-0.103312,0.009613,-0.003763,0.036917,30.38
2,269,15824187,-0.002654,0.083027,0.016132,0.056085,16.09
3,771,23143190,0.022156,-0.076737,-0.024679,0.061093,23.54
4,1024,29487120,0.10182,-0.009797,0.005491,0.039161,29.99


**Lectura.** Con la referencia nacional la entidad explica la mayor parte de `brecha` y `tendencia` y varias entidades quedan casi enteras en un perfil. Con la referencia por entidad ese efecto desaparece, y es la que usan las secciones 4 a 6.

## 8. Calidad por sección: NS (spec 002, RF-01)

`NS` no es abstención: son registros de cuadernillos sin marcas, no disponibles, solicitados por autoridad jurisdiccional o con páginas faltantes. Se marcan las secciones cuyo `NS` supera el 25 % de su lista en una elección (D1). La corrida principal no cambia; la sección 9 la repite sin esas secciones.

In [15]:
ns = analisis.paso_calidad_ns()
ns["resumen"]

AELEC,secciones,secciones_marcadas,LN,LN_marcada,pct_secciones,pct_LN
i16,u32,u32,i64,i64,f64,f64
2009,64934,2444,77481831,2846581,3.76,3.67
2012,65599,251,76490962,286040,0.38,0.37
2015,68362,6313,83563462,8086724,9.23,9.68
2018,68408,7415,89123997,9566865,10.84,10.73
2021,68812,5078,93532133,6333824,7.38,6.77
2024,70751,7193,98330348,9964539,10.17,10.13


## 9. Sensibilidad y robustez (spec 002, RF-02 a RF-05 y RF-13)

Se repite k-means con otros datos o rasgos (sin secciones con NS alto, con volatilidad como cuarto rasgo y con la circunscripción como territorio) y se compara con otros algoritmos sobre los mismos datos (Ward y mezcla gaussiana, sin pesos). El ARI mide la coincidencia con la asignación principal: 1 es idéntica y 0 es al azar.

In [16]:
for corrida in ["ns", "region"]:
    analisis.paso_indicadores(corrida)
for corrida in ["ns", "volatilidad", "region"]:
    analisis.paso_segmentacion(corrida)
    analisis.paso_perfiles(corrida)
analisis.paso_comparar_corridas()

corrida,k,celdas_principal,celdas_comparadas,pct_LN_comparada,ARI_vs_principal
str,i64,i64,i64,f64,f64
"""nacional""",4,2841,2841,100.0,0.121746
"""ns""",5,2841,2841,100.0,0.930193
"""volatilidad""",4,2841,2841,100.0,0.444505
"""region""",5,2841,2841,100.0,0.321816


In [17]:
rob = analisis.paso_robustez()
print(rob["pca"]["varianza"]); print(rob["pca"]["cargas"]); print(rob["bic"])
rob["comparacion"]

shape: (3, 3)
┌────────────┬────────────────────┬───────────┐
│ componente ┆ varianza_explicada ┆ acumulada │
│ ---        ┆ ---                ┆ ---       │
│ str        ┆ f64                ┆ f64       │
╞════════════╪════════════════════╪═══════════╡
│ PC1        ┆ 0.428438           ┆ 0.428438  │
│ PC2        ┆ 0.332068           ┆ 0.760506  │
│ PC3        ┆ 0.239494           ┆ 1.0       │
└────────────┴────────────────────┴───────────┘
shape: (3, 4)
┌───────────┬───────────┬──────────┬───────────┐
│ rasgo     ┆ PC1       ┆ PC2      ┆ PC3       │
│ ---       ┆ ---       ┆ ---      ┆ ---       │
│ str       ┆ f64       ┆ f64      ┆ f64       │
╞═══════════╪═══════════╪══════════╪═══════════╡
│ nivel     ┆ -0.368709 ┆ 0.85802  ┆ 0.357569  │
│ brecha    ┆ 0.704718  ┆ 0.007177 ┆ 0.709451  │
│ tendencia ┆ 0.606157  ┆ 0.513566 ┆ -0.607308 │
└───────────┴───────────┴──────────┴───────────┘
shape: (3, 2)
┌─────┬──────────────┐
│ k   ┆ BIC_mezcla   │
│ --- ┆ ---          │
│ i64 ┆ f64     

metodo,k,usa_pesos,ARI_vs_kmeans
str,i64,bool,f64
"""Ward (jerárquico)""",5,false,0.373588
"""Mezcla gaussiana""",5,false,0.258361


**Lectura.** Quitar las secciones con NS alto casi no cambia los perfiles. Los otros algoritmos y la variante con volatilidad coinciden solo en parte: las celdas no forman grupos muy separados (silhouette cercano a 0.24), así que los perfiles son una partición útil de un continuo y no tipos naturales. El perfil 1 (jóvenes urbanos) es el que más se conserva entre métodos; ver `robustez_cruce_*.csv` y la figura `12_cruce_metodos.png`.

## 10. Cohortes y primer voto (spec 002, RF-06 a RF-08 y RF-15)

Una cohorte agrupa a quienes nacieron el mismo año y se observa en cada elección: es un pseudo-panel, no sigue a las mismas personas. El año de nacimiento es aproximado (±1 año) y los efectos de edad, periodo y cohorte no se separan por completo. Los perfiles y las cohortes describen grupos y territorios; atribuir sus rasgos a una persona sería una falacia ecológica.

In [18]:
co = analisis.paso_cohortes()
co["primer_voto"]

nacimiento,primera_eleccion,edad_primera,rel_primera,LN_primera,rel_siguiente,cambio_siguiente,rel_seis_anios,cambio_seis_anios
i16,i16,i16,f64,i64,f64,f64,f64,f64
1989,2009,20,-0.065891,1877722,-0.09505,-0.029159,-0.125279,-0.059388
1990,2009,19,-0.032272,1722398,-0.090558,-0.058285,-0.123772,-0.0915
1991,2009,18,0.046865,900133,-0.077668,-0.124533,-0.118895,-0.16576
1992,2012,20,-0.062614,2003906,-0.117946,-0.055332,-0.099634,-0.03702
1993,2012,19,-0.031856,1855530,-0.113903,-0.082047,-0.102318,-0.070463
…,…,…,…,…,…,…,…,…
2002,2021,19,-0.033402,1904469,-0.115304,-0.081901,null,null
2003,2021,18,0.037624,1318719,-0.108271,-0.145895,null,null
2004,2024,20,-0.093592,2193418,null,null,null,null


## 11. Distritos electorales federales (spec 002, RF-09 a RF-12)

De 2009 a 2021 el distrito federal viene en el marco cartográfico 2021, así que es comparable entre elecciones. En 2024 viene en el marco 2023; para la trayectoria, cada sección de 2024 recibe su distrito de 2021. Las secciones nuevas de 2024 quedan «sin distrito» (D4). La composición de perfiles se da en los dos marcos: 2023 para uso actual y 2021 para comparar.

In [19]:
ter = analisis.paso_territorio()
print(ter["conservacion"])
ter["circunscripcion"]

shape: (6, 6)
┌───────┬─────────────────┬─────────────────┬───────────┬────────┬──────────────────┐
│ AELEC ┆ LN_con_distrito ┆ LN_sin_distrito ┆ LN_fuente ┆ cuadra ┆ pct_sin_distrito │
│ ---   ┆ ---             ┆ ---             ┆ ---       ┆ ---    ┆ ---              │
│ i16   ┆ i64             ┆ i64             ┆ i64       ┆ bool   ┆ f64              │
╞═══════╪═════════════════╪═════════════════╪═══════════╪════════╪══════════════════╡
│ 2009  ┆ 75958987        ┆ 1522844         ┆ 77481831  ┆ true   ┆ 1.965            │
│ 2012  ┆ 75581039        ┆ 909923          ┆ 76490962  ┆ true   ┆ 1.19             │
│ 2015  ┆ 83563462        ┆ 0               ┆ 83563462  ┆ true   ┆ 0.0              │
│ 2018  ┆ 89123997        ┆ 0               ┆ 89123997  ┆ true   ┆ 0.0              │
│ 2021  ┆ 93532133        ┆ 0               ┆ 93532133  ┆ true   ┆ 0.0              │
│ 2024  ┆ 93143416        ┆ 5186932         ┆ 98330348  ┆ true   ┆ 5.275            │
└───────┴─────────────────┴─────────────

circunscripcion,LN,pct_perfil_1,pct_perfil_2,pct_perfil_3,pct_perfil_4,pct_perfil_5
i8,i64,f64,f64,f64,f64,f64
1,20513006,34.44535,8.655659,30.531329,13.024975,13.342687
2,19105662,31.462408,9.303693,36.370813,11.190461,11.672626
3,18932822,32.747855,12.775392,22.445121,15.846189,16.185443
4,20516230,35.881154,6.61338,28.661932,9.501639,19.341895
5,19259731,33.548478,10.044741,32.122606,13.250963,11.033212


## 12. Estadísticas y figuras de apoyo (spec 002, RF-16)

Figuras en `entrega/estadisticas/`: distribuciones de los datos, algoritmos, cohortes y distritos. Cada una lee una tabla de `entrega/tablas/`.

In [20]:
import estadisticas
analisis.paso_descriptivas()
estadisticas.generar_todo()

['01_piramide_lista_nominal_2024.png',
 '02_participacion_por_tipo_seccion.png',
 '03_participacion_por_entidad.png',
 '04_participacion_por_edad.png',
 '05_distribucion_ns_por_seccion.png',
 '06_distribucion_rasgos_celdas.png',
 '07_seleccion_k.png',
 '08_pca_varianza_cargas.png',
 '09_pca_perfiles.png',
 '10_rasgos_por_perfil.png',
 '11_robustez_metodos.png',
 '12_cruce_metodos.png',
 '13_cohortes_trayectoria.png',
 '14_efecto_primer_voto.png',
 '15_cohortes_por_perfil.png',
 '16_distritos_rasgos.png',
 '17_distritos_trayectoria.png',
 '18_perfiles_por_circunscripcion.png',
 '19_distritos_perfil_dominante.png']

## 13. Entregables

In [21]:
import contenido, documento, figuras, presentacion, verificacion

figuras.generar_todo()
ENTREGA = TABLAS.parent
documento.construir(ENTREGA / "documento_tecnico.pdf")
presentacion.construir(ENTREGA / "presentacion_ejecutiva.pptx")
verificacion.resumen()

{'paginas': 10,
 'fuentes': {'LiberationSans', 'LiberationSans-Bold', 'LiberationSans-Italic'},
 'diapositivas': 10,
 'fichas': 5,
 'dpi_fichas': [300, 300, 300, 300, 300],
 'rayas_largas': 0}

Los marcadores `[NOMBRE DEL EQUIPO]` y las cartas firmadas (RE-05 y RE-06) quedan para el equipo. Ver `entrega/lista_verificacion.md`.